# Experiment 6: k-Nearest Neighbor (k-NN) - Iris Dataset

## Overview
k-Nearest Neighbor is a simple, instance-based lazy learning algorithm that classifies new instances based on the k closest training examples. It requires no training phase and makes predictions based on distance metrics.

## Key Concepts
- **Instance-based Learning**: Stores all training examples
- **Lazy Learning**: No explicit training phase
- **Distance Metrics**: Euclidean distance (most common), Manhattan, etc.
- **Parameter k**: Number of nearest neighbors to consider
- **Majority Voting**: Final class is determined by majority vote among k neighbors

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

print("k-Nearest Neighbor (k-NN) Algorithm")
print("=" * 60)

## Load and Prepare Data

In [ ]:
# Load iris dataset
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print("\nIris Dataset:")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Features: {feature_names}")
print(f"Classes: {target_names}")
print(f"Class distribution: {np.bincount(y)}")

# Create DataFrame for visualization
df = pd.DataFrame(X, columns=feature_names)
df['Species'] = df.index.map(lambda i: target_names[y[i]])
print("\nFirst 5 rows:")
print(df.head())

## Data Splitting and Preprocessing

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nClass distribution in training set: {np.bincount(y_train)}")
print(f"Class distribution in test set: {np.bincount(y_test)}")

# Standardize features (important for distance-based algorithms)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeatures standardized.")

## Find Optimal k

In [ ]:
# Test different values of k
k_values = range(1, 21)
train_scores = []
test_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_scores.append(knn.score(X_train_scaled, y_train))
    test_scores.append(knn.score(X_test_scaled, y_test))

# Find optimal k
optimal_k = list(k_values)[test_scores.index(max(test_scores))]
print(f"Optimal k value: {optimal_k} (Test accuracy: {max(test_scores):.4f})")

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(k_values, train_scores, marker='o', label='Training Accuracy', linewidth=2)
ax.plot(k_values, test_scores, marker='s', label='Test Accuracy', linewidth=2)
ax.axvline(x=optimal_k, color='red', linestyle='--', alpha=0.5, label=f'Optimal k={optimal_k}')
ax.set_xlabel('k (Number of Neighbors)')
ax.set_ylabel('Accuracy')
ax.set_title('k-NN Accuracy vs k Value')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(range(1, 21))
plt.tight_layout()
plt.show()

## Train k-NN with Optimal k

In [ ]:
# Train k-NN with optimal k
knn_classifier = KNeighborsClassifier(n_neighbors=optimal_k)
knn_classifier.fit(X_train_scaled, y_train)

# Predictions
y_train_pred = knn_classifier.predict(X_train_scaled)
y_test_pred = knn_classifier.predict(X_test_scaled)

# Accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("\n" + "="*60)
print(f"k-NN CLASSIFIER (k={optimal_k})")
print("="*60)
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## Classification Report

In [ ]:
print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_test_pred, target_names=target_names))

# Performance metrics
precision = precision_score(y_test, y_test_pred, average='weighted')
recall = recall_score(y_test, y_test_pred, average='weighted')
f1 = f1_score(y_test, y_test_pred, average='weighted')

print(f"\nWeighted Metrics:")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

## Confusion Matrix

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
ax.set_title(f'Confusion Matrix - k-NN Classifier (k={optimal_k})')
plt.tight_layout()
plt.show()

## Correct and Incorrect Predictions

In [ ]:
# Analyze predictions
print("\n" + "="*60)
print("PREDICTION ANALYSIS")
print("="*60)

correct_mask = y_test == y_test_pred
incorrect_mask = y_test != y_test_pred

print(f"\nTotal Test Samples: {len(y_test)}")
print(f"Correct Predictions: {np.sum(correct_mask)}")
print(f"Incorrect Predictions: {np.sum(incorrect_mask)}")
print(f"\nAccuracy: {np.sum(correct_mask)/len(y_test):.4f}")

# Print correct predictions
print("\n--- CORRECT PREDICTIONS ---")
correct_count = 0
for i in np.where(correct_mask)[0][:10]:  # Show first 10
    print(f"Sample {i}: True={target_names[y_test[i]]}, "
          f"Predicted={target_names[y_test_pred[i]]} ✓")
    correct_count += 1
if np.sum(correct_mask) > 10:
    print(f"... and {np.sum(correct_mask) - 10} more correct predictions")

# Print incorrect predictions
print("\n--- INCORRECT PREDICTIONS ---")
incorrect_count = 0
for i in np.where(incorrect_mask)[0]:
    print(f"Sample {i}: True={target_names[y_test[i]]}, "
          f"Predicted={target_names[y_test_pred[i]]} ✗")
    incorrect_count += 1
if incorrect_count == 0:
    print("No incorrect predictions!")

## Visualization of Nearest Neighbors

In [ ]:
# Get distances and indices of k nearest neighbors for first test sample
distances, indices = knn_classifier.kneighbors(X_test_scaled[:1])

print(f"\nNearest {optimal_k} neighbors for first test sample:")
print(f"Test sample true class: {target_names[y_test[0]]}")
print(f"Test sample predicted class: {target_names[y_test_pred[0]]}\n")

for i, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
    neighbor_class = target_names[y_train[idx]]
    print(f"Neighbor {i}: Class={neighbor_class}, Distance={dist:.4f}")

## Summary Statistics

In [ ]:
# Plot summary
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy comparison
ax = axes[0, 0]
metrics_names = ['Train', 'Test']
accuracies = [train_accuracy, test_accuracy]
colors = ['green', 'blue']
bars = ax.bar(metrics_names, accuracies, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Accuracy')
ax.set_ylim([0, 1])
ax.set_title('Accuracy Comparison')
ax.grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{acc:.4f}', ha='center', va='bottom')

# Per-class accuracy
ax = axes[0, 1]
class_accuracies = []
for class_idx in range(len(target_names)):
    mask = y_test == class_idx
    if np.sum(mask) > 0:
        class_acc = np.sum((y_test[mask] == y_test_pred[mask])) / np.sum(mask)
        class_accuracies.append(class_acc)
bars = ax.bar(target_names, class_accuracies, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_ylabel('Accuracy')
ax.set_ylim([0, 1])
ax.set_title('Per-Class Accuracy')
ax.grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, class_accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{acc:.4f}', ha='center', va='bottom')

# Prediction distribution
ax = axes[1, 0]
pred_counts = np.bincount(y_test_pred, minlength=len(target_names))
true_counts = np.bincount(y_test, minlength=len(target_names))
x = np.arange(len(target_names))
width = 0.35
ax.bar(x - width/2, true_counts, width, label='True', alpha=0.7, edgecolor='black')
ax.bar(x + width/2, pred_counts, width, label='Predicted', alpha=0.7, edgecolor='black')
ax.set_ylabel('Count')
ax.set_title('Class Distribution')
ax.set_xticks(x)
ax.set_xticklabels(target_names)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Metrics
ax = axes[1, 1]
metric_names = ['Precision', 'Recall', 'F1-Score', 'Accuracy']
metric_values = [precision, recall, f1, test_accuracy]
colors_metrics = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
bars = ax.bar(metric_names, metric_values, color=colors_metrics, alpha=0.7, edgecolor='black')
ax.set_ylabel('Score')
ax.set_ylim([0, 1])
ax.set_title('Performance Metrics')
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, metric_values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{val:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Viva Questions

### Basic Concepts
1. **What is k-Nearest Neighbor algorithm?**
   - Instance-based lazy learning algorithm
   - Classifies new samples based on k closest training examples
   - Uses distance metrics to find neighbors

2. **What does "lazy" learning mean in k-NN?**
   - No explicit training phase
   - All computation happens at prediction time
   - Training data is stored as-is

3. **What is the role of parameter k?**
   - Number of nearest neighbors to consider
   - Small k: more sensitive to noise, may overfit
   - Large k: smoother decision boundaries, may underfit

### Algorithm Questions
4. **How does k-NN make predictions?**
   - Find k nearest training samples using distance metric
   - Use majority voting among neighbors' classes
   - Assign most frequent class to new sample

5. **What distance metrics are commonly used?**
   - Euclidean distance: √(Σ(x_i - y_i)²)
   - Manhattan distance: Σ|x_i - y_i|
   - Minkowski distance: (Σ|x_i - y_i|^p)^(1/p)
   - Choice affects classification results

6. **Why is feature scaling important in k-NN?**
   - Features with larger scale dominate distance calculation
   - StandardScaler makes all features contribute equally
   - Improves model performance significantly

### Practical Questions
7. **What are advantages of k-NN?**
   - Simple and easy to understand
   - Works well with multi-class problems
   - No assumptions about data distribution
   - Can adapt to new data easily

8. **What are disadvantages of k-NN?**
   - Computationally expensive at prediction time (O(nd))
   - Sensitive to feature scaling
   - Sensitive to choice of k and distance metric
   - Struggles with high-dimensional data

9. **How do you choose the value of k?**
   - Usually use cross-validation
   - Test different k values and choose best
   - Odd k for binary classification (avoid ties)
   - Rule of thumb: k = √n (n = training samples)

10. **How does k-NN handle imbalanced datasets?**
    - Minority class may be ignored if k is large
    - Can weight neighbors by inverse distance
    - Can use stratified sampling
    - Consider cost-sensitive learning